In [2]:
import pandas as pd

In [3]:
stk_data=pd.read_csv("TATACOFEE_1321.csv")

In [4]:
stk_data

,Date,Price,Open,High,Low,Vol.,Change %
0,15-01-2013,"1,586.95","1,605.45","1,605.45","1,582.20",57.64K,-0.42%
1,16-01-2013,"1,572.70","1,588.05","1,591.15","1,570.00",62.35K,-0.90%
2,17-01-2013,"1,599.90","1,577.15","1,645.80","1,570.00",225.98K,1.73%
3,18-01-2013,"1,587.00","1,609.80","1,614.70","1,578.55",71.61K,-0.81%
4,21-01-2013,"1,596.90","1,594.75","1,621.00","1,587.95",64.27K,0.62%
...,...,...,...,...,...,...,...
2210,27-12-2021,218.35,200.00,222.00,196.00,5.89M,8.63%
2211,28-12-2021,212.35,219.65,220.45,211.55,2.87M,-2.75%
2212,29-12-2021,211.35,213.00,216.70,210.00,2.71M,-0.47%
2213,30-12-2021,208.50,211.45,211.50,207.90,977.48K,-1.35%


In [5]:
stk_data.rename(columns={"Price":"Close"}, inplace=True)

In [6]:
stk_data["Date"]=pd.to_datetime(stk_data["Date"],dayfirst=True)

In [7]:
stk_data=stk_data[(stk_data["Date"]>="2020-07-01")&(stk_data["Date"]<="2021-12-31")]

In [8]:
stk_data=stk_data.sort_values("Date")
stk_data=stk_data.set_index("Date")

In [9]:
stk_data

,Close,Open,High,Low,Vol.,Change %
Date,,,,,,
2020-07-01,81.95,81.50,82.70,81.05,318.89K,0.00%
2020-07-02,81.90,82.25,82.70,81.55,267.38K,-0.06%
2020-07-03,84.35,82.05,85.45,82.05,921.18K,2.99%
2020-07-06,86.60,85.80,87.85,85.05,1.66M,2.67%
2020-07-07,85.65,86.70,87.00,85.05,476.14K,-1.10%
...,...,...,...,...,...,...
2021-12-27,218.35,200.00,222.00,196.00,5.89M,8.63%
2021-12-28,212.35,219.65,220.45,211.55,2.87M,-2.75%
2021-12-29,211.35,213.00,216.70,210.00,2.71M,-0.47%


In [10]:
# preprocessing
from sklearn.preprocessing import MinMaxScaler
Ms=MinMaxScaler()
data1=Ms.fit_transform(stk_data[["Close"]])
print("Len:",data1.shape)

Len: (376, 1)


In [20]:
# Creating the model
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
def holtwinters_model(data1):
    test_obs = 28
    # train and test split
    train=data1[:-test_obs]
    test=data1[-test_obs:]
    # Holt-winters model
    model=ExponentialSmoothing(
           train,
           trend="add",
           seasonal="add",
           seasonal_periods=5,
           initialization_method="estimated"
    )
    result=model.fit()
    # prediction
    y_pred=result.forecast(test_obs)
    # RMSE
    rmse= round(mean_squared_error(test,y_pred)**0.5,4)
    # MAPE
    mape=mean_absolute_percentage_error(test,y_pred)
    print("RMSE:",rmse)
    print("MAPE:", mape)
    return y_pred,result,rmse,mape

In [21]:
y_pred_hw, result_hw, rmse_hw, mape_hw = holtwinters_model(data1)

RMSE: 0.099
MAPE: 0.10988462357286768


In [23]:
from stockFunctions import conversionOriginal
forecast_stock_price_oriF=conversionOriginal(y_pred_hw,["CloseFore"],Ms)

In [24]:
forecast_stock_price_oriF

,CloseFore
0,217.098552
1,217.112319
2,217.471746
3,218.555303
4,219.236026
5,219.034578
6,219.048345
7,219.407772
8,220.491329
9,221.172052


In [25]:
forecast_stock_price_oriF.to_csv("HES_Closet.csv", index=False)